In [ ]:
!pip install sentence-transformers stanza vaderSentiment scikit-learn pandas numpy torch transformers
!pip install --upgrade  sentence-transformers 

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!ls "/content/drive/MyDrive/Colab_Notebooks/a5"

Mounted at /content/drive
result_v8.csv		 train_2022.csv  vBERT_finetune.csv
test_no_answer_2022.csv  v8_result


In [3]:
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import stanza
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader, Dataset as TorchDataset
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, mean_squared_error,
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score,
)
from sklearn.pipeline import Pipeline
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [11]:
base_path = "/content/drive/MyDrive/Colab_Notebooks/a5/"
CFG = {
    # I/O
    "input":        base_path + "train_2022.csv",
    "output":       base_path + "row_scores_v8_no_vader_no_nlp.csv",
    "test":         base_path + "test_no_answer_2022.csv",
    "test_output":  base_path + "result_v8_no_vader_no_nlp.csv",

    # ML model: "rf" | "elastic" | "gbm"
    "model":        "rf",

    # Embedding weight scheme: "sqrt" | "uniform" | "linear" | "log" | "decay" | "last" | "contrast"
    "weight":       "sqrt",

    # Phase 1 — MLM Domain Adaptation
    "skip_da":      False,
    "load_da":      None,
    "da_epochs":    1,
    "da_batch":     8,
    "da_lr":        3e-5,
    "da_mlm_prob":  0.15,
    "da_save":      base_path + "v8_da_model.pt",

    # Phase 1 only mode
    "skip_ft":      False,

    # Phase 2 — Supervised Fine-tuning
    "ft_epochs":    3,
    "ft_lr":        2e-5,
    "ft_batch":     16,
    "ft_pairs":     3000,
    "ft_warmup":    100,
    "ft_save":      base_path + "v8_ft_model.pt",
}

SBERT_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
SBERT_DIM        = 768

# nlp (stanza) and vader temporarily disabled
VADER_FEATURES = []
AGG_FEATURES   = [f"emb_{i}" for i in range(SBERT_DIM)]
ALL_FEATURES   = AGG_FEATURES

WEIGHT_SCHEMES = ("sqrt", "uniform", "linear", "log", "decay", "last", "contrast")

TRANSITION_WORDS = {
    "but", "however", "although", "though", "yet", "nevertheless",
    "nonetheless", "except", "while", "whereas",
}

In [5]:
c = CFG
print(c["input"])
!ls {c['load_da']}

/content/drive/MyDrive/Colab_Notebooks/a5/train_2022.csv
config.json			   model.safetensors  README.md
config_sentence_transformers.json  modules.json       tokenizer.json


In [6]:
def _build_sbert_model() -> SentenceTransformer:
    print(f"  Loading base Sentence-BERT '{SBERT_MODEL_NAME}' …")
    model = SentenceTransformer(SBERT_MODEL_NAME)
    print(f"  Embedding dim : {SBERT_DIM}")
    return model


class _TextDataset(TorchDataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}


def domain_adapt_mlm(
    sbert: SentenceTransformer,
    all_texts: list[str],
    epochs: int     = 1,
    batch_size: int = 16,
    lr: float       = 3e-5,
    mlm_prob: float = 0.15,
    output_path: str | None = None,
) -> None:
    from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling

    print(f"\n── Phase 1: MLM Domain Adaptation ────────────────────────")
    print(f"  Texts    : {len(all_texts)} (Train + Test, no labels)")
    print(f"  Epochs   : {epochs}  |  batch={batch_size}  |  lr={lr}")
    print(f"  MLM prob : {mlm_prob} (隨機遮蔽 {int(mlm_prob*100)}% tokens)")

    transformer_module = sbert[0]
    hf_model_name      = transformer_module.auto_model.config.name_or_path

    tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
    mlm_model = AutoModelForMaskedLM.from_pretrained(hf_model_name)

    mlm_model.mpnet.load_state_dict(
        transformer_module.auto_model.state_dict(), strict=False
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mlm_model.to(device)
    mlm_model.train()

    encodings = tokenizer(
        all_texts,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt",
    )
    dataset    = _TextDataset(encodings)
    collator   = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True,
                                                  mlm_probability=mlm_prob)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                            collate_fn=collator)

    optimizer    = torch.optim.AdamW(mlm_model.parameters(), lr=lr)
    total_steps  = len(dataloader) * epochs
    warmup_steps = max(1, total_steps // 10)

    scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_steps
    )

    global_step = 0
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for step, batch in enumerate(dataloader, 1):
            batch  = {k: v.to(device) for k, v in batch.items()}
            output = mlm_model(**batch)
            loss   = output.loss
            loss.backward()
            optimizer.step()
            if global_step < warmup_steps:
                scheduler.step()
            optimizer.zero_grad()
            total_loss  += loss.item()
            global_step += 1
            if step % 50 == 0:
                print(f"    epoch {epoch}/{epochs}  step {step}/{len(dataloader)}"
                      f"  loss={total_loss/step:.4f}")
        print(f"  Epoch {epoch} done — avg loss: {total_loss/len(dataloader):.4f}")

    mlm_model.to("cpu")
    transformer_module.auto_model.load_state_dict(
        mlm_model.mpnet.state_dict(), strict=False
    )

    if output_path:
        sbert.save(output_path)
        print(f"  Saved adapted model to: {output_path}")

    print("  Phase 1 complete — encoder adapted to domain vocabulary.\n")

In [7]:
def _sample_pairs(df: pd.DataFrame, n_pairs: int, seed: int = 42) -> list[InputExample]:
    rng    = random.Random(seed)
    texts  = df["TEXT"].astype(str).tolist()
    labels = df["LABEL"].astype(int).tolist()

    pos_idx = [i for i, l in enumerate(labels) if l == 1]
    neg_idx = [i for i, l in enumerate(labels) if l == 0]

    examples = []
    half     = n_pairs // 2

    for _ in range(half):
        pool = pos_idx if rng.random() < 0.5 else neg_idx
        if len(pool) >= 2:
            a, b = rng.sample(pool, 2)
        else:
            a = b = pool[0]
        examples.append(InputExample(texts=[texts[a], texts[b]], label=1.0))

    for _ in range(n_pairs - half):
        a = rng.choice(pos_idx)
        b = rng.choice(neg_idx)
        examples.append(InputExample(texts=[texts[a], texts[b]], label=0.0))

    rng.shuffle(examples)
    return examples


def finetune_sbert_supervised(
    sbert: SentenceTransformer,
    train_df: pd.DataFrame,
    n_pairs: int    = 3000,
    epochs: int     = 3,
    batch_size: int = 16,
    lr: float       = 2e-5,
    warmup_steps: int = 100,
    output_path: str | None = None,
) -> None:
    print(f"\n── Phase 2: Supervised Fine-tuning (CosineSimilarityLoss) ──")
    print(f"  Train rows  : {len(train_df)}  |  pairs={n_pairs}")
    print(f"  Epochs      : {epochs}  |  batch={batch_size}  |  lr={lr}")
    print(f"  Strategy    : same label→sim=1.0, diff label→sim=0.0")

    examples   = _sample_pairs(train_df, n_pairs)
    dataloader = DataLoader(examples, shuffle=True, batch_size=batch_size,
                            collate_fn=lambda b: b)
    loss_fn    = losses.CosineSimilarityLoss(sbert)

    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sbert.to(device)
    sbert.train()

    optimizer   = torch.optim.AdamW(sbert.parameters(), lr=lr)
    total_steps = len(dataloader) * epochs
    scheduler   = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0,
        total_iters=min(warmup_steps, total_steps)
    )

    global_step = 0
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for step, batch in enumerate(dataloader, 1):
            texts_a  = [ex.texts[0] for ex in batch]
            texts_b  = [ex.texts[1] for ex in batch]
            labels   = torch.tensor([ex.label for ex in batch],
                                    dtype=torch.float, device=device)

            feats_a  = {k: v.to(device) for k, v in sbert.tokenize(texts_a).items()
                        if isinstance(v, torch.Tensor)}
            feats_b  = {k: v.to(device) for k, v in sbert.tokenize(texts_b).items()
                        if isinstance(v, torch.Tensor)}

            loss = loss_fn([feats_a, feats_b], labels)
            loss.backward()
            optimizer.step()
            if global_step < warmup_steps:
                scheduler.step()
            optimizer.zero_grad()
            total_loss  += loss.item()
            global_step += 1
            if step % 50 == 0:
                print(f"    epoch {epoch}/{epochs}  step {step}/{len(dataloader)}"
                      f"  loss={total_loss/step:.4f}")
        print(f"  Epoch {epoch} done — avg loss: {total_loss/len(dataloader):.4f}")

    sbert.train(False)
    if output_path:
        sbert.save(output_path)
        print(f"  Saved fine-tuned model to: {output_path}")
    print("  Phase 2 complete — model now clusters same-sentiment sentences.\n")

In [8]:
def _compute_weights(n: int, scheme: str) -> list:
    if scheme == "uniform":  return [1.0] * n
    if scheme == "linear":   return [float(i + 1) for i in range(n)]
    if scheme == "sqrt":     return [np.sqrt(i + 1) for i in range(n)]
    if scheme == "log":      return [np.log(i + 2) for i in range(n)]
    if scheme == "decay":    return [np.exp(-0.5 * i) for i in range(n)]
    if scheme == "last":
        w = [1.0] * n
        if n > 0: w[-1] = 3.0
        return w
    if scheme == "contrast":
        if n == 1: return [2.0]
        w = [0.5] * n; w[0] = 2.0; w[-1] = 2.0
        return w
    raise ValueError(f"Unknown weight scheme '{scheme}'.")


def _get_embedding(sbert: SentenceTransformer, text: str) -> np.ndarray:
    emb = sbert.encode(text, normalize_embeddings=True, show_progress_bar=False)
    return emb.astype(np.float32)


def process_row(text: str, label: int, row_id: int,
                sbert: SentenceTransformer,
                weight_scheme: str = "sqrt") -> dict:
    # nlp/vader disabled: encode the full text as a single segment
    segments = [text]
    n        = len(segments)
    weights  = _compute_weights(n, weight_scheme)

    agg_emb = np.zeros(SBERT_DIM, dtype=np.float32)
    for i, seg in enumerate(segments):
        agg_emb += _get_embedding(sbert, seg) * weights[i]

    result = {"row_id": row_id, "LABEL": label, "NUM_SPLITS": n}
    for j, val in enumerate(agg_emb):
        result[f"emb_{j}"] = round(float(val), 6)
    return result


def run_row_by_row(df: pd.DataFrame,
                   sbert: SentenceTransformer,
                   has_label: bool = True,
                   weight_scheme: str = "sqrt") -> pd.DataFrame:
    total = len(df)
    print(f"  Extracting embeddings for {total} rows …")

    records = []
    for i, (_, row) in enumerate(df.iterrows()):
        if i % 200 == 0:
            print(f"    progress: {i}/{total}")
        label = int(row["LABEL"]) if has_label else -1
        records.append(
            process_row(str(row["TEXT"]), label, int(row["row_id"]),
                        sbert, weight_scheme=weight_scheme)
        )

    cols      = ["row_id", "LABEL", "NUM_SPLITS"] + AGG_FEATURES
    result_df = pd.DataFrame(records)[cols]
    print(f"  Done — {len(result_df)} rows embedded  (feature dim: {len(ALL_FEATURES)})")
    return result_df

In [9]:
def _make_model(model_name: str):
    model_map = {
        "rf":      RandomForestRegressor(n_estimators=100, random_state=42),
        "elastic": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000),
        "gbm":     GradientBoostingRegressor(n_estimators=100, random_state=42),
    }
    if model_name not in model_map:
        raise ValueError(f"Unknown model '{model_name}'. Choose from: {list(model_map)}")
    return model_map[model_name]


def _print_fold_metrics(fold: int, y_true, y_pred_label, y_score) -> dict:
    cm  = confusion_matrix(y_true, y_pred_label)
    acc = accuracy_score(y_true, y_pred_label)
    pre = precision_score(y_true, y_pred_label, zero_division=0)
    rec = recall_score(y_true, y_pred_label, zero_division=0)
    f1  = f1_score(y_true, y_pred_label, zero_division=0)
    mse = mean_squared_error(y_true, y_score)
    try:
        auc = roc_auc_score(y_true, y_score)
    except Exception:
        auc = float("nan")

    print(f"\n  ── Fold {fold} ──────────────────────────────")
    print(f"  Confusion Matrix:\n{cm}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {pre:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")
    print(f"  AUC      : {auc:.4f}  MSE: {mse:.4f}")
    return dict(acc=acc, pre=pre, rec=rec, f1=f1, auc=auc, mse=mse, cm=cm)


def run_cross_validation(agg_df: pd.DataFrame, model_name: str, n_splits: int = 5):
    print(f"  Model: {model_name}  |  Features: {SBERT_DIM}-dim SBERT + 4 VADER = {len(ALL_FEATURES)}-dim")
    print(f"  Running {n_splits}-fold stratified cross-validation …")

    X = agg_df[ALL_FEATURES].values
    y = agg_df["LABEL"].values

    skf          = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_metrics = []
    oof_scores   = np.zeros(len(y))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
        pipe = Pipeline([("scaler", StandardScaler()), ("model", _make_model(model_name))])
        pipe.fit(X[train_idx], y[train_idx])
        scores              = np.clip(pipe.predict(X[val_idx]), 0, 1)
        oof_scores[val_idx] = scores
        fold_metrics.append(_print_fold_metrics(fold, y[val_idx], (scores >= 0.5).astype(int), scores))

    oof_labels = (oof_scores >= 0.5).astype(int)
    print("\n  ══ Overall OOF (out-of-fold) ═══════════════════════")
    print(f"  Confusion Matrix:\n{confusion_matrix(y, oof_labels)}")
    for metric in ("acc", "pre", "rec", "f1", "auc", "mse"):
        vals = [m[metric] for m in fold_metrics]
        print(f"  {metric.upper():9s}: mean={np.mean(vals):.4f}  std={np.std(vals):.4f}")

    print("\n  Retraining on full dataset …")
    final_pipe = Pipeline([("scaler", StandardScaler()), ("model", _make_model(model_name))])
    final_pipe.fit(X, y)
    final_scores = np.clip(final_pipe.predict(X), 0, 1)

    output = agg_df[["row_id", "LABEL"]].copy().rename(columns={"LABEL": "row_label"})
    output["oof_score"]       = oof_scores
    output["oof_pred_label"]  = oof_labels
    output["final_score"]     = final_scores
    output["predicted_label"] = (final_scores >= 0.5).astype(int)
    return output, final_pipe, ALL_FEATURES

In [12]:
def main():
    c = CFG

    sep = "=" * 60
    print(f"\n{sep}")
    print("  Sentiment Scoring Pipeline  [v8 / no-arg]")
    print("  Two-Phase Fine-Tuning: MLM → CosineSimilarityLoss")
    print("  [nlp / vader temporarily disabled]")
    print(sep)
    print(f"  Input      : {c['input']}")
    print(f"  Test       : {c['test']}")
    print(f"  Model      : {c['model']}")
    print(f"  Encoder    : {SBERT_MODEL_NAME}  ({SBERT_DIM}-dim)")
    if c["load_da"]:
        phase1_desc = f"LOAD from {c['load_da']}"
    elif c["skip_da"]:
        phase1_desc = "SKIPPED"
    else:
        phase1_desc = f"MLM  epochs={c['da_epochs']}  lr={c['da_lr']}  mlm_prob={c['da_mlm_prob']}"
    print(f"  Phase 1    : {phase1_desc}")
    print(f"  Phase 2    : CosineSimilarityLoss  epochs={c['ft_epochs']}  lr={c['ft_lr']}  pairs={c['ft_pairs']}")
    print(f"  Weighting  : {c['weight']}\n")

    train_df = pd.read_csv(c["input"])
    train_df["row_id"] = train_df["row_id"].astype(int)
    train_df["LABEL"]  = train_df["LABEL"].astype(int)
    print(f"  Loaded {len(train_df)} training rows")

    test_df = pd.read_csv(c["test"])
    test_df["row_id"] = test_df["row_id"].astype(int)
    if "LABEL" not in test_df.columns:
        test_df["LABEL"] = -1
    print(f"  Loaded {len(test_df)} test rows\n")

    print("── Initialising models ─────────────────────────────────────")
    sbert = _build_sbert_model()

    if c["load_da"]:
        print(f"\n  [Phase 1] 載入已存 checkpoint: {c['load_da']}\n")
        sbert = SentenceTransformer(c["load_da"])
    elif not c["skip_da"]:
        all_texts = (
            train_df["TEXT"].astype(str).tolist() +
            test_df["TEXT"].astype(str).tolist()
        )
        domain_adapt_mlm(
            sbert, all_texts,
            epochs=c["da_epochs"], batch_size=c["da_batch"],
            lr=c["da_lr"], mlm_prob=c["da_mlm_prob"],
            output_path=c["da_save"],
        )
    else:
        print("\n  [Phase 1 skipped — using base model weights]\n")

    if c["skip_ft"]:
        print("  [skip_ft=True，Phase 1 完成後退出]")
        return

    finetune_sbert_supervised(
        sbert, train_df,
        n_pairs=c["ft_pairs"], epochs=c["ft_epochs"],
        batch_size=c["ft_batch"], lr=c["ft_lr"],
        warmup_steps=c["ft_warmup"], output_path=c["ft_save"],
    )

    print("── Steps 1+2: Split + Embedding (two-phase fine-tuned SBERT) ──")
    agg_df = run_row_by_row(train_df, sbert,
                            has_label=True, weight_scheme=c["weight"])

    print("\n── Step 3: 5-Fold CV + Regression Scoring ──────────────────")
    row_scores, trained_pipe, feat_cols = run_cross_validation(agg_df, c["model"], n_splits=5)

    # row_scores.to_csv(c["output"], index=False)
    # print(f"\n  Saved training results to: {c['output']}")

    print(f"\n── Processing test set ─────────────────────────────────────")
    test_agg_df = run_row_by_row(test_df, sbert,
                                 has_label=False, weight_scheme=c["weight"])

    X_test      = test_agg_df[feat_cols].values
    test_scores = np.clip(trained_pipe.predict(X_test), 0, 1)
    test_labels = (test_scores >= 0.5).astype(int)

    test_output = test_agg_df[["row_id"]].copy()
    test_output["LABEL"] = test_labels
    test_output.to_csv(c["test_output"], index=False)
    print(f"  Saved test predictions to: {c['test_output']}")


main()


  Sentiment Scoring Pipeline  [v8 / no-arg]
  Two-Phase Fine-Tuning: MLM → CosineSimilarityLoss
  [nlp / vader temporarily disabled]
  Input      : /content/drive/MyDrive/Colab_Notebooks/a5/train_2022.csv
  Test       : /content/drive/MyDrive/Colab_Notebooks/a5/test_no_answer_2022.csv
  Model      : rf
  Encoder    : sentence-transformers/all-mpnet-base-v2  (768-dim)
  Phase 1    : MLM  epochs=1  lr=3e-05  mlm_prob=0.15
  Phase 2    : CosineSimilarityLoss  epochs=3  lr=2e-05  pairs=3000
  Weighting  : sqrt

  Loaded 2000 training rows
  Loaded 11000 test rows

── Initialising models ─────────────────────────────────────
  Loading base Sentence-BERT 'sentence-transformers/all-mpnet-base-v2' …
  Embedding dim : 768

── Phase 1: MLM Domain Adaptation ────────────────────────
  Texts    : 13000 (Train + Test, no labels)
  Epochs   : 1  |  batch=8  |  lr=3e-05
  MLM prob : 0.15 (隨機遮蔽 15% tokens)


Some weights of the model checkpoint at sentence-transformers/all-mpnet-base-v2 were not used when initializing MPNetForMaskedLM: ['pooler.dense.bias', 'pooler.dense.weight']
- This IS expected if you are initializing MPNetForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing MPNetForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of MPNetForMaskedLM were not initialized from the model checkpoint at sentence-transformers/all-mpnet-base-v2 and are newly initialized: ['lm_head.bias', 'lm_head.decoder.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
You should probably TRAIN this model on a down-stream task to be 

    epoch 1/1  step 50/1625  loss=10.4581
    epoch 1/1  step 100/1625  loss=9.3528
    epoch 1/1  step 150/1625  loss=8.6202
    epoch 1/1  step 200/1625  loss=8.1310
    epoch 1/1  step 250/1625  loss=7.7619
    epoch 1/1  step 300/1625  loss=7.4519
    epoch 1/1  step 350/1625  loss=7.2156
    epoch 1/1  step 400/1625  loss=6.9989
    epoch 1/1  step 450/1625  loss=6.7973
    epoch 1/1  step 500/1625  loss=6.6371
    epoch 1/1  step 550/1625  loss=6.5103
    epoch 1/1  step 600/1625  loss=6.3759
    epoch 1/1  step 650/1625  loss=6.2706
    epoch 1/1  step 700/1625  loss=6.1707
    epoch 1/1  step 750/1625  loss=6.0684
    epoch 1/1  step 800/1625  loss=5.9675
    epoch 1/1  step 850/1625  loss=5.8961
    epoch 1/1  step 900/1625  loss=5.8289
    epoch 1/1  step 950/1625  loss=5.7500
    epoch 1/1  step 1000/1625  loss=5.7037
    epoch 1/1  step 1050/1625  loss=5.6495
    epoch 1/1  step 1100/1625  loss=5.6013
    epoch 1/1  step 1150/1625  loss=5.5524
    epoch 1/1  step 1200/1625 

  Saved adapted model to: /content/drive/MyDrive/Colab_Notebooks/a5/v8_da_model.pt
  Phase 1 complete — encoder adapted to domain vocabulary.


── Phase 2: Supervised Fine-tuning (CosineSimilarityLoss) ──
  Train rows  : 2000  |  pairs=3000
  Epochs      : 3  |  batch=16  |  lr=2e-05
  Strategy    : same label→sim=1.0, diff label→sim=0.0
    epoch 1/3  step 50/188  loss=0.2850
    epoch 1/3  step 100/188  loss=0.2680
    epoch 1/3  step 150/188  loss=0.2503
  Epoch 1 done — avg loss: 0.2386
    epoch 2/3  step 50/188  loss=0.1280
    epoch 2/3  step 100/188  loss=0.1300
    epoch 2/3  step 150/188  loss=0.1214
  Epoch 2 done — avg loss: 0.1155
    epoch 3/3  step 50/188  loss=0.0570
    epoch 3/3  step 100/188  loss=0.0533
    epoch 3/3  step 150/188  loss=0.0540
  Epoch 3 done — avg loss: 0.0530
  Saved fine-tuned model to: /content/drive/MyDrive/Colab_Notebooks/a5/v8_ft_model.pt
  Phase 2 complete — model now clusters same-sentiment sentences.

── Steps 1+2: Split + Embedding (two-ph